# Cancellation also runs cleanup

Cancellation asks an asynchronous task to stop. It is a separate exit path from a normal result or an ordinary exception.

Use the repository Python environment and run the cells in order. This notebook has its own setup; no previous notebook is required.

## 1. Import the implementation

The lifecycle code comes from AOA itself. The example supplies only the resource or external connection used to observe it.

In [1]:
import asyncio

from aoa.action_machine.graph.core.exclude_graph_model import exclude_graph_model
from aoa.action_machine.resources.base_resource import BaseResource
from aoa.action_machine.resources.connection_scope import connection_scope
from aoa.action_machine.resources.per_call_connection import PerCallConnection

## 2. Define the objects used in this case

These definitions give the case an observable result without introducing unrelated business behavior.

In [2]:
@exclude_graph_model
class TraceResource(BaseResource):
    """Record lifecycle calls without opening an external connection."""
    def __init__(self, events):
        """Keep the event list supplied by the example."""
        self.events = events
    def get_wrapper_class(self):
        """Allow direct use by nested actions."""
        return None
    async def open(self):
        """Record acquisition."""
        self.events.append("open")
    async def close(self):
        """Record release."""
        self.events.append("close")

## 3. Prepare this call

Create the state and declarations used below. Merely declaring a per-call factory does not run it.

In [3]:
events = []
started = asyncio.Event()
async def work():
    """Wait until the caller cancels this task."""
    declaration = PerCallConnection(lambda: TraceResource(events), lambda resource: resource.close())
    async with connection_scope({"db": declaration}):
        started.set()
        await asyncio.Event().wait()
task = asyncio.create_task(work())

## 4. Run and check the observation

The assertions test this case’s stated behavior. Printed output is recorded below; expected failures are caught explicitly.

In [4]:
await started.wait()
task.cancel()
try:
    await task
except asyncio.CancelledError:
    print("CancelledError")
assert events == ["open", "close"]
print(events)

CancelledError
['open', 'close']


## 5. Explain the result

The caller still receives CancelledError, but only after the finalizer has run. No delay or guessed scheduling interval is needed: the event ensures acquisition happened before cancellation.